In [1]:
import pandas as pd
df=pd.read_csv('data/loan_approval_dataset.csv')
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [2]:
df.columns=df.columns.str.strip()

In [3]:
df.columns

Index(['loan_id', 'no_of_dependents', 'education', 'self_employed',
       'income_annum', 'loan_amount', 'loan_term', 'cibil_score',
       'residential_assets_value', 'commercial_assets_value',
       'luxury_assets_value', 'bank_asset_value', 'loan_status'],
      dtype='object')

In [4]:
X=df.iloc[:,1:-1].drop(columns='loan_amount')
X.head()

,no_of_dependents,education,self_employed,income_annum,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
0,2,Graduate,No,9600000,12,778,2400000,17600000,22700000,8000000
1,0,Not Graduate,Yes,4100000,8,417,2700000,2200000,8800000,3300000
2,3,Graduate,No,9100000,20,506,7100000,4500000,33300000,12800000
3,3,Graduate,No,8200000,8,467,18200000,3300000,23300000,7900000
4,5,Not Graduate,Yes,9800000,20,382,12400000,8200000,29400000,5000000


In [32]:
X['education'].value_counts()

education
Graduate        2144
Not Graduate    2125
Name: count, dtype: int64

In [35]:
X['education'].unique()

array([' Graduate', ' Not Graduate'], dtype=object)

In [5]:
X.columns

Index(['no_of_dependents', 'education', 'self_employed', 'income_annum',
       'loan_term', 'cibil_score', 'residential_assets_value',
       'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value'],
      dtype='object')

In [7]:
y=df.iloc[:,5]
y.head()

0    29900000
1    12200000
2    29700000
3    30700000
4    24200000
Name: loan_amount, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error,mean_absolute_error
import pickle


In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=True,random_state=42)

In [12]:
X_train.shape,X_test.shape

((3415, 10), (854, 10))

In [13]:
cat_cols=X.select_dtypes(include='O').columns
num_cols=X.select_dtypes(include='number').columns

In [14]:
cat_cols

Index(['education', 'self_employed'], dtype='object')

In [15]:
num_cols

Index(['no_of_dependents', 'income_annum', 'loan_term', 'cibil_score',
       'residential_assets_value', 'commercial_assets_value',
       'luxury_assets_value', 'bank_asset_value'],
      dtype='object')

In [16]:
onehot=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
std=StandardScaler()

preprocessing=ColumnTransformer([
    ('catcolumns',onehot,cat_cols),
    ('numcolumns',std,num_cols)
])
preprocessing

,transformers,"[('catcolumns', ...), ('numcolumns', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [17]:
lr=LinearRegression()

model=Pipeline([
    ('preprocessing',preprocessing),
    ('modelling',lr)
])
model

,steps,"[('preprocessing', ...), ('modelling', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('catcolumns', ...), ('numcolumns', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
model.fit(X_train,y_train)

# testing
test_results=model.predict(X_test)
test_mae=mean_absolute_error(y_test,test_results)
test_mse=mean_squared_error(y_test,test_results)
print(f"Mean absolute error for testing results : {test_mae: .2f}")
print(f"Mean squared error for testing results : {test_mse: .2f}")

Mean absolute error for testing results :  2626034.54
Mean squared error for testing results :  11915621630392.53


In [22]:
#training

train_results=model.predict(X_train)
train_mae=mean_absolute_error(y_train,train_results)
train_mse=mean_squared_error(y_train,train_results)
print(f"Mean absolute error for testing results : {train_mae: .2f}")
print(f"Mean squared error for testing results : {train_mse: .2f}")

Mean absolute error for testing results :  2563779.09
Mean squared error for testing results :  11288473739036.84


In [24]:
with open('loan_prediction.pkl','wb') as f:
    pickle.dump(model,f)

In [25]:
X_train.shape,X_test.shape

((3415, 10), (854, 10))

In [26]:
X_test.head()

,no_of_dependents,education,self_employed,income_annum,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
1703,5,Graduate,No,5400000,20,423,6500000,10000000,15700000,7300000
1173,2,Graduate,No,5900000,8,599,4700000,9500000,17800000,6700000
308,3,Graduate,No,9600000,14,452,4200000,16200000,28500000,6600000
1322,2,Graduate,No,6200000,8,605,10000000,10800000,21800000,9200000
3271,3,Not Graduate,Yes,5800000,12,738,11700000,4400000,15400000,8400000


In [27]:
test_data=X_test.loc[1173,:]
test_data

no_of_dependents                    2
education                    Graduate
self_employed                      No
income_annum                  5900000
loan_term                           8
cibil_score                       599
residential_assets_value      4700000
commercial_assets_value       9500000
luxury_assets_value          17800000
bank_asset_value              6700000
Name: 1173, dtype: object

In [28]:
type(test_data)

pandas.core.series.Series

In [29]:
test_data=test_data.to_dict()
test_data

{'no_of_dependents': 2,
 'education': ' Graduate',
 'self_employed': ' No',
 'income_annum': 5900000,
 'loan_term': 8,
 'cibil_score': 599,
 'residential_assets_value': 4700000,
 'commercial_assets_value': 9500000,
 'luxury_assets_value': 17800000,
 'bank_asset_value': 6700000}

In [30]:
test_dataframe=pd.DataFrame([test_data])
test_dataframe

,no_of_dependents,education,self_employed,income_annum,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
0,2,Graduate,No,5900000,8,599,4700000,9500000,17800000,6700000


In [31]:
model.predict(test_dataframe)

array([17713157.6357988])